# 어린이 과학영어 라디오 - 리틀사이언스팟

## 목적:
부모가 장거리 이동이나 대기 시간 동안 아이에게 계속 영상만 보여주지 않아도 되도록, 귀로 듣는 과학 영어 콘텐츠를 만들어 주는 교육용 에이전트입니다.
아이가 지루하지 않게 짧은 이야기, 쉬운 과학 개념, 반복 영어 표현, 캐릭터 대화를 섞은 키즈 라디오 스크립트를 만들고, 이를 캐릭터 목소리 오디오 파일로 생성하는 것이 목표입니다.

## 핵심 기능
- 어린이 맞춤 과학영어 스크립트 생성연령대, 주제, 길이, 분위기를 입력받아 라디오 형식 스크립트를 생성합니다.
- 캐릭터 기반 대화형 구성호기심 많은 아이와 설명해 주는 박사님 캐릭터를 자동 배치합니다.
- 단순 설명문이 아니라 대화극처럼 구성해 아이가 더 잘 듣게 합니다.
- 쉬운 과학 + 쉬운 영어 변환어려운 과학 개념을 아이 눈높이로 풀고, 핵심 영어 표현을 반복해서 넣습니다.
- 멀티 보이스 오디오 생성스크립트를 캐릭터별로 나누어 각기 다른 목소리로 연기한 오디오 파일을 만듭니다.
- 최종적으로 하나의 라디오 에피소드로 합칩니다.
- 부모용 학습 요약 제공


## 그래프 구조

```mermaid
graph TD
    A[부모 입력<br/>연령대, 주제, 길이, 분위기] --> B[에피소드 기획 노드]
    B --> C[어린이 과학영어 스크립트 생성]
    C --> D[사용자 검토]
    D -->|수정 요청| C
    D -->|승인| E[스크립트 확정]

    E --> F[캐릭터 배치<br/>아이, 박사님]
    F --> G[대화극 구성]
    G --> H[쉬운 과학 설명 변환]
    H --> I[핵심 영어 표현 반복 삽입]

    I --> J[캐릭터별 대사 분리]
    J --> K1[아이 음성 생성]
    J --> K2[박사님 음성 생성]

    K1 --> L[멀티 보이스 오디오 합성]
    K2 --> L

    L --> M[최종 라디오 에피소드<br/>MP3/WAV]
    E --> N[부모용 학습 요약]
```


In [ ]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class EpisodeState(TypedDict, total=False):
    age_group: str
    topic: str
    duration_minutes: int
    tone: str
    plan: str
    script: str
    parent_summary: str


In [ ]:
def plan_episode(state: EpisodeState) -> EpisodeState:
    age_group = state["age_group"]
    topic = state["topic"]
    duration = state["duration_minutes"]
    tone = state["tone"]

    plan = (
        f"- Audience: {age_group}\n"
        f"- Topic: {topic}\n"
        f"- Length: about {duration} minutes\n"
        f"- Tone: {tone}\n"
        "- Characters: Curious Kid Leo, Dr. Spark\n"
        "- Learning goal: explain one science idea with short repeated English phrases\n"
        "- Repeated phrase: 'Look up!', 'Can you say it?', 'Science is fun!'"
    )
    return {"plan": plan}


def write_script(state: EpisodeState) -> EpisodeState:
    topic = state["topic"]

    script = f"""
[Leo] Wow! I want to know about {topic}!

[Dr. Spark] Great question, Leo. {topic.capitalize()} can be explained in a simple way.
[Dr. Spark] We look, we ask, and we learn. Science is fun!

[Leo] Look up! Can you say it? Look up!

[Leo] I learned one new idea today!
[Dr. Spark] See you next time on Little Science Spot!
""".strip()
    return {"script": script}


def summarize_for_parent(state: EpisodeState) -> EpisodeState:
    topic = state["topic"]

    parent_summary = (
        f"Today's episode introduced '{topic}' with a short character-based dialogue. "
        "Children hear repeated expressions such as 'Look up!' and 'Science is fun!' "
        "to support listening and speaking practice."
    )
    return {"parent_summary": parent_summary}


In [ ]:
graph_builder = StateGraph(EpisodeState)

graph_builder.add_node("plan_episode", plan_episode)
graph_builder.add_node("write_script", write_script)
graph_builder.add_node("summarize_for_parent", summarize_for_parent)

graph_builder.add_edge(START, "plan_episode")
graph_builder.add_edge("plan_episode", "write_script")
graph_builder.add_edge("write_script", "summarize_for_parent")
graph_builder.add_edge("summarize_for_parent", END)

graph = graph_builder.compile()
graph


In [ ]:
result = graph.invoke(
    {
        "age_group": "6-8",
        "topic": "the moon",
        "duration_minutes": 3,
        "tone": "playful",
    }
)

result
